In [1]:
# 1. Instalar librerías si hace falta
!pip install plotly openpyxl -q

In [2]:
# 2. Importar librerías
import pandas as pd
import plotly.express as px

In [11]:
ruta_riesgo = "/content/tabla_riesgo_unidades_filtrado_solo_1200_y_atraso600 (1).xlsx"
ruta_unidades = "/content/new_unidades (2).xlsx"

df_riesgo = pd.read_excel(
    ruta_riesgo,
    sheet_name="Tabla_Riesgo_Final"
)

df_unidades = pd.read_excel(ruta_unidades)

# Limpiar espacios raros en nombres de columnas
df_riesgo.columns = df_riesgo.columns.astype(str).str.strip()
df_unidades.columns = df_unidades.columns.astype(str).str.strip()

print("Columnas riesgo:")
print(df_riesgo.columns.tolist())

print("\nColumnas unidades:")
print(df_unidades.columns.tolist())

Columnas riesgo:
['Nivel de riesgo', 'Score riesgo', 'Alias', 'No. Serie', 'Marca', 'Modelo', 'Distribuidor', 'Estado', 'Horómetro', 'Edad equipo', 'Total servicios', 'Cerrados', 'Cerrada fuera', 'Pendientes', 'Backlog', 'Atraso máx hrs', 'Atraso prom hrs', 'T. Encendido hrs', 'Distancia', 'Última fecha mant.', 'Motivo clasificación']

Columnas unidades:
['IMEI', 'Alias', 'Marca', 'Modelo', 'Configuracion', 'Fecha Alta', 'No. Serie', 'Distribuidor', 'Ciudad', 'Estado', 'Horometro', 'Odometro', 'Total', 'Cerrados', 'C.Fuera', 'Pendientes', '50', '300', '600', '900', '1200', '1500', '1800', '2100', '2400', 'Latitud', 'Longitud', 'Combustible', 'Distancia', 'Vel. Maxima (kph)', 'Vel. Promedio (kph)', 'T. Encendido (hrs)', 'T. Apagado (hrs)', 'intervalos_con_info', 'intervalos_sin_info', 'edad_equipo']


In [5]:
# 4. Revisar columnas disponibles
df.columns

Index(['Nivel de riesgo', 'Score riesgo', 'Alias', 'No. Serie', 'Marca',
       'Modelo', 'Distribuidor', 'Estado', 'Horómetro', 'Edad equipo',
       'Total servicios', 'Cerrados', 'Cerrada fuera', 'Pendientes', 'Backlog',
       'Atraso máx hrs', 'Atraso prom hrs', 'T. Encendido hrs', 'Distancia',
       'Última fecha mant.', 'Motivo clasificación'],
      dtype='object')

In [12]:
# Nos quedamos con las columnas de riesgo que necesitamos
riesgo_cols = [
    "Alias",
    "Nivel de riesgo",
    "Score riesgo",
    "Pendientes",
    "Cerrada fuera",
    "Backlog",
    "Atraso máx hrs",
    "Horómetro"
]

df_riesgo_base = df_riesgo[riesgo_cols].copy()

# Nos quedamos con ubicación desde new_unidades
ubicacion_cols = [
    "Alias",
    "Latitud",
    "Longitud",
    "Ciudad",
    "Estado",
    "Distribuidor"
]

df_ubicacion = df_unidades[ubicacion_cols].copy()

# Cruzar por Alias
df_mapa = df_riesgo_base.merge(
    df_ubicacion,
    on="Alias",
    how="left"
)

df_mapa.head()

,Alias,Nivel de riesgo,Score riesgo,Pendientes,Cerrada fuera,Backlog,Atraso máx hrs,Horómetro,Latitud,Longitud,Ciudad,Estado,Distribuidor
0,TAPSAR-H20358M-353738103189443,Crítico,80.0,4,1,5,535,8635.7,18.660283,-97.633355,Tepanco de Lopez,Puebla,TAPSA
1,MADISA - S306522M-860092059729422,Crítico,79.4,17,1,18,1148,1198.1,18.013197,-95.851259,San Juan Bautista Tuxtepec,Oaxaca,MADISAGRAL
2,7610S 2WD-5,Crítico,78.3,19,8,27,2381,0.0,23.899022,-106.928809,Desconocido,Desconocido,Desconocido
3,TAPSA - S306541M,Crítico,78.2,9,0,9,1106,1155.6,18.556395,-97.514371,Tepanco de Lopez,Puebla,TAPSA
4,TRACSA - S306820M,Crítico,77.7,5,0,5,1149,1198.9,19.318696,-101.514125,Tacambaro,Michoacan de Ocampo,TRACSAGRAL


In [13]:
df_mapa["Latitud"] = pd.to_numeric(df_mapa["Latitud"], errors="coerce")
df_mapa["Longitud"] = pd.to_numeric(df_mapa["Longitud"], errors="coerce")

df_mapa = df_mapa.dropna(subset=["Latitud", "Longitud"]).copy()

df_mapa = df_mapa[
    (df_mapa["Latitud"] != 0) &
    (df_mapa["Longitud"] != 0)
]

print("Unidades de riesgo con ubicación válida:", len(df_mapa))
df_mapa.head()

Unidades de riesgo con ubicación válida: 3825


,Alias,Nivel de riesgo,Score riesgo,Pendientes,Cerrada fuera,Backlog,Atraso máx hrs,Horómetro,Latitud,Longitud,Ciudad,Estado,Distribuidor
0,TAPSAR-H20358M-353738103189443,Crítico,80.0,4,1,5,535,8635.7,18.660283,-97.633355,Tepanco de Lopez,Puebla,TAPSA
1,MADISA - S306522M-860092059729422,Crítico,79.4,17,1,18,1148,1198.1,18.013197,-95.851259,San Juan Bautista Tuxtepec,Oaxaca,MADISAGRAL
2,7610S 2WD-5,Crítico,78.3,19,8,27,2381,0.0,23.899022,-106.928809,Desconocido,Desconocido,Desconocido
3,TAPSA - S306541M,Crítico,78.2,9,0,9,1106,1155.6,18.556395,-97.514371,Tepanco de Lopez,Puebla,TAPSA
4,TRACSA - S306820M,Crítico,77.7,5,0,5,1149,1198.9,19.318696,-101.514125,Tacambaro,Michoacan de Ocampo,TRACSAGRAL


In [14]:
mapa_agrupado = (
    df_mapa
    .groupby(
        ["Estado", "Ciudad", "Distribuidor", "Latitud", "Longitud", "Nivel de riesgo"],
        dropna=False
    )
    .agg(
        unidades=("Alias", "count"),
        pendientes=("Pendientes", "sum"),
        cerrada_fuera=("Cerrada fuera", "sum"),
        backlog=("Backlog", "sum"),
        score_promedio=("Score riesgo", "mean"),
        atraso_maximo=("Atraso máx hrs", "max")
    )
    .reset_index()
)

mapa_agrupado["servicios_oportunidad"] = (
    mapa_agrupado["pendientes"] + mapa_agrupado["cerrada_fuera"]
)

mapa_agrupado.head()

,Estado,Ciudad,Distribuidor,Latitud,Longitud,Nivel de riesgo,unidades,pendientes,cerrada_fuera,backlog,score_promedio,atraso_maximo,servicios_oportunidad
0,Aguascalientes,Aguascalientes,EMEJORES,21.801272,-102.280595,Alto,1,3,0,3,44.5,271,3
1,Aguascalientes,Aguascalientes,TEPSA,21.814356,-102.362438,Alto,1,2,1,3,42.4,227,3
2,Aguascalientes,Aguascalientes,TRACSAGRAL,21.636383,-102.312917,Alto,1,4,0,4,70.8,877,4
3,Aguascalientes,Aguascalientes,TRACSAGRAL,21.642798,-102.102206,Alto,1,3,0,3,49.3,469,3
4,Aguascalientes,Aguascalientes,TRACSAGRAL,21.720479,-102.119642,Alto,1,4,0,4,63.5,431,4


In [26]:
niveles_prioritarios = ["Crítico", "Alto"]

mapa_prioritario = mapa_agrupado[
    mapa_agrupado["Nivel de riesgo"].astype(str).str.strip().isin(niveles_prioritarios)
].copy()

colores_riesgo = {
    "Crítico": "#20235C",   # azul
    "Alto": "#B45309"       # naranja
}

fig = px.scatter_mapbox(
    mapa_prioritario,
    lat="Latitud",
    lon="Longitud",
    size="unidades",
    color="Nivel de riesgo",
    color_discrete_map=colores_riesgo,
    category_orders={
        "Nivel de riesgo": ["Crítico", "Alto"]
    },
    hover_name="Ciudad",
    hover_data={
        "Estado": True,
        "Distribuidor": True,
        "Nivel de riesgo": True,
        "unidades": True,
        "pendientes": True,
        "cerrada_fuera": True,
        "servicios_oportunidad": True,
        "backlog": True,
        "score_promedio": ":.1f",
        "atraso_maximo": True,
        "Latitud": False,
        "Longitud": False
    },
    zoom=4.7,
    center={"lat": 23.6345, "lon": -102.5528},
    mapbox_style="carto-positron",
    title="Concentración geográfica de unidades críticas y altas",
    height=700
)

fig.update_traces(
    marker=dict(
        opacity=0.9
    )
)

fig.update_layout(
    paper_bgcolor="#F9FAFB",
    plot_bgcolor="#FFFFFF",
    font=dict(
        family="Arial",
        size=14,
        color="#111827"
    ),
    title=dict(
        font=dict(size=22, color="#111827"),
        x=0.02
    ),
    legend=dict(
        title="Nivel de riesgo",
        bgcolor="#FFFFFF",
        bordercolor="#E5E7EB",
        borderwidth=1,
        font=dict(color="#111827")
    ),
    margin={"r":20, "t":60, "l":20, "b":20}
)

fig.show()

In [27]:
# Top 3 estados donde más conviene enfocar operación
niveles_prioritarios = ["Crítico", "Alto"]

df_prioritario = df_riesgo[
    df_riesgo["Nivel de riesgo"].astype(str).str.strip().isin(niveles_prioritarios)
].copy()

top_estados = (
    df_prioritario
    .groupby("Estado")
    .agg(
        unidades_prioritarias=("Nivel de riesgo", "size"),
        criticas=("Nivel de riesgo", lambda x: (x == "Crítico").sum()),
        altas=("Nivel de riesgo", lambda x: (x == "Alto").sum()),
        pendientes=("Pendientes", "sum"),
        cerrada_fuera=("Cerrada fuera", "sum")
    )
    .reset_index()
)

top_estados["servicios_oportunidad"] = (
    top_estados["pendientes"] + top_estados["cerrada_fuera"]
)

top_3_estados = top_estados.sort_values(
    by=["unidades_prioritarias", "criticas", "servicios_oportunidad"],
    ascending=False
).head(3)

top_3_estados

,Estado,unidades_prioritarias,criticas,altas,pendientes,cerrada_fuera,servicios_oportunidad
14,Jalisco,387,220,167,4539,320,4859
21,Puebla,320,196,124,4728,141,4869
11,Guanajuato,237,152,85,2324,374,2698
